In [3]:
import requests
import json
import pandas as pd


In [4]:
def buscar_incorporadoras_ativas_dataframe(uf=None, limit=100):
    """
    Busca CNPJs ativos de incorporadoras (CNAE 4110-7/00) usando a API Minha Receita
    e retorna os resultados em um DataFrame do Pandas.

    Args:
        uf (str, optional): Sigla do estado para filtrar (ex: 'SP', 'RJ'). 
                            Se None, busca em todo o Brasil.
        limit (int, optional): Quantidade de resultados por página (máx 1000).

    Returns:
        pd.DataFrame: DataFrame do Pandas contendo os dados das empresas ativas.
    """
    base_url = "https://minhareceita.org/"
    cnae_incorporadora = "4110700" # CNAE sem pontuação
    
    params = {
        "cnae": cnae_incorporadora,
        "limit": limit
    }
    if uf:
        params["uf"] = uf
        
    print(f"Buscando incorporadoras na API (UF: {uf if uf else 'Todas'}, Limite: {limit})...")
    
    empresas_ativas_data = []
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()

        if 'data' in data and isinstance(data['data'], list):
            cnpjs_encontrados = data['data']
            
            # Filtrar apenas as empresas com situação cadastral ATIVA (código 2)
            for empresa in cnpjs_encontrados:
                if empresa.get('situacao_cadastral') == 2: # 2 = ATIVA
                    empresas_ativas_data.append(empresa)
                    
            print(f"Total de resultados na página: {len(cnpjs_encontrados)}")
            print(f"Total de empresas ATIVAS encontradas: {len(empresas_ativas_data)}")
            
            if empresas_ativas_data:
                df = pd.DataFrame(empresas_ativas_data)
                # Selecionar e renomear colunas relevantes para melhor visualização
                colunas_selecionadas = [
                    'cnpj', 'razao_social', 'nome_fantasia', 
                    'descricao_situacao_cadastral', 'uf', 'municipio', 
                    'cnae_fiscal_descricao', 'data_inicio_atividade'
                ]
                df_final = df[colunas_selecionadas].copy()
                df_final.rename(columns={
                    'cnpj': 'CNPJ',
                    'razao_social': 'Razão Social',
                    'nome_fantasia': 'Nome Fantasia',
                    'descricao_situacao_cadastral': 'Situação Cadastral',
                    'uf': 'UF',
                    'municipio': 'Município',
                    'cnae_fiscal_descricao': 'CNAE Fiscal',
                    'data_inicio_atividade': 'Data Início Atividade'
                }, inplace=True)
                return df_final
            else:
                return pd.DataFrame() # Retorna um DataFrame vazio se não houver empresas ativas
        else:
            print("Nenhum dado encontrado ou formato inesperado.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"Erro de conexão com a API: {e}")
        return pd.DataFrame()

In [5]:
if __name__ == "__main__":
    # Exemplo: Buscar incorporadoras ativas em São Paulo e Rio de Janeiro
    df_sp = buscar_incorporadoras_ativas_dataframe(uf="SP", limit=10)
    if not df_sp.empty:
        print("\n--- Incorporadoras Ativas em São Paulo ---")
        print(df_sp.head())
        # Exemplo de como salvar em CSV ou Excel
        # df_sp.to_csv("incorporadoras_sp.csv", index=False)
        # df_sp.to_excel("incorporadoras_sp.xlsx", index=False)
    else:
        print("\nNenhuma incorporadora ativa encontrada em São Paulo.")

    df_rj = buscar_incorporadoras_ativas_dataframe(uf="RJ", limit=10)
    if not df_rj.empty:
        print("\n--- Incorporadoras Ativas no Rio de Janeiro ---")
        print(df_rj.head())
    else:
        print("\nNenhuma incorporadora ativa encontrada no Rio de Janeiro.")

    # Exemplo de busca sem UF (todo o Brasil)
    # df_br = buscar_incorporadoras_ativas_dataframe(limit=10)
    # if not df_br.empty:
    #     print("\n--- Incorporadoras Ativas no Brasil (Primeiros 10) ---")
    #     print(df_br.head())

Buscando incorporadoras na API (UF: SP, Limite: 10)...
Total de resultados na página: 10
Total de empresas ATIVAS encontradas: 8

--- Incorporadoras Ativas em São Paulo ---
             CNPJ                                       Razão Social  \
0  24503934000376         SPE AXIS 1 EMPREENDIMENTO IMOBILIARIO LTDA   
1  18758523000167          ALISEOS EMPREENDIMENTOS IMOBILIARIOS LTDA   
2  32774477000110                      EUGENIO DE CAMARGO LEITE LTDA   
3  17243059000103  GIANOTTO & LEAO EMPREENDIMENTOS IMOBILIARIOS LTDA   
4  17242004000189                               KWL CONSTRUCOES LTDA   

                     Nome Fantasia Situação Cadastral  UF  \
0                                               ATIVA  SP   
1                                               ATIVA  SP   
2                                               ATIVA  SP   
3  GIANOTTO & LEAO EMPREENDIMENTOS              ATIVA  SP   
4                                               ATIVA  SP   

               Município   

In [1]:
import requests
import pandas as pd
import time
import re


def classificar_empresa(nome):
    """
    Classifica empresa como SPE ou Incorporadora provável
    """
    nome = (nome or "").upper()

    padroes_spe = [
        "SPE",
        "EMPREENDIMENTO",
        "IMOBILIARIO",
        "RESIDENCIAL",
        "LTDA SCP"
    ]

    if any(p in nome for p in padroes_spe):
        return "SPE_PROVAVEL"

    return "INCORPORADORA_PROVAVEL"


def buscar_incorporadoras_ativas_dataframe(uf=None, limite_total=500):
    base_url = "https://minhareceita.org/"
    cnae = "4110700"

    resultados = []
    offset = 0
    limite_pagina = 100  # pode aumentar (até 1000 dependendo da API)

    print(f"Buscando dados (UF={uf}, limite_total={limite_total})...")

    while len(resultados) < limite_total:
        params = {
            "cnae": cnae,
            "limit": limite_pagina,
            "offset": offset
        }

        if uf:
            params["uf"] = uf

        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            empresas = data.get("data", [])
            if not empresas:
                break

            for emp in empresas:
                if emp.get("situacao_cadastral") == 2:
                    emp["tipo_empresa"] = classificar_empresa(
                        emp.get("razao_social") or emp.get("nome_fantasia")
                    )
                    resultados.append(emp)

            print(f"Coletados: {len(resultados)}")

            offset += limite_pagina
            time.sleep(0.5)  # evitar rate limit

        except Exception as e:
            print(f"Erro: {e}")
            break

    if not resultados:
        return pd.DataFrame()

    df = pd.DataFrame(resultados)

    # 🔥 Limpeza e enriquecimento
    colunas = [
        'cnpj', 'razao_social', 'nome_fantasia',
        'municipio', 'uf',
        'cnae_fiscal_descricao',
        'data_inicio_atividade',
        'tipo_empresa'
    ]

    df = df[colunas].copy()

    df.rename(columns={
        'cnpj': 'CNPJ',
        'razao_social': 'Razão Social',
        'nome_fantasia': 'Nome Fantasia',
        'municipio': 'Município',
        'uf': 'UF',
        'cnae_fiscal_descricao': 'CNAE',
        'data_inicio_atividade': 'Início',
        'tipo_empresa': 'Classificação'
    }, inplace=True)

    return df

In [2]:
df = buscar_incorporadoras_ativas_dataframe(uf="PR", limite_total=500)

# Separar grupos
df_spe = df[df["Classificação"] == "SPE_PROVAVEL"]
df_real = df[df["Classificação"] == "INCORPORADORA_PROVAVEL"]

print("SPE:", len(df_spe))
print("Incorporadoras reais:", len(df_real))

Buscando dados (UF=PR, limite_total=500)...
Coletados: 69
Coletados: 138
Coletados: 207
Coletados: 276
Coletados: 345
Coletados: 414
Coletados: 483
Coletados: 552
SPE: 160
Incorporadoras reais: 392


In [10]:
df_real["Classificação"].unique()

<StringArray>
['INCORPORADORA_PROVAVEL']
Length: 1, dtype: str

In [11]:
df_spe

,CNPJ,Razão Social,Nome Fantasia,Município,UF,CNAE,Início,Classificação
3,42908452000116,OKABE EMPREENDIMENTOS IMOBILIARIOS SPE LTDA,,CASCAVEL,PR,Loteamento de imóveis próprios,2021-07-29,SPE_PROVAVEL
4,17276840000184,MAYLE - EMPREENDIMENTOS IMOBILIARIOS LTDA,,APUCARANA,PR,Construção de edifícios,2012-11-28,SPE_PROVAVEL
8,31486143000260,TOOWOOMBA INCORPORACOES SPE 4 LTDA,RESIDENCIAL EVORA,LONDRINA,PR,Incorporação de empreendimentos imobiliários,2021-04-26,SPE_PROVAVEL
14,02380176000141,GREEN FIELDS EMPREENDIMENTOS IMOBILIARIOS LTDA,SPECIAL RESIDENCE,CURITIBA,PR,Incorporação de empreendimentos imobiliários,1998-02-10,SPE_PROVAVEL
15,17314830000196,SPE PROJETO 3 LTDA.,SPE PROJETO 3 LTDA.,CURITIBA,PR,Incorporação de empreendimentos imobiliários,2012-09-18,SPE_PROVAVEL
...,...,...,...,...,...,...,...,...
535,18986833000139,CUBO RESIDENCE RUBY EMPREENDIMENTOS IMOBILIARI...,,CURITIBA,PR,Incorporação de empreendimentos imobiliários,2013-09-13,SPE_PROVAVEL
540,30202787000135,J&MM EMPREENDIMENTOS IMOBILIARIOS LTDA,J&MM EMPREENDIMENTOS IMOBILIARIOS,CURITIBA,PR,Incorporação de empreendimentos imobiliários,2018-03-06,SPE_PROVAVEL
545,43134960000157,CONCREALFA EMPREENDIMENTOS IMOBILIARIOS LTDA,CONCREALFA EMPREENDIMENTOS IMOBILIARIOS LTDA,PONTA GROSSA,PR,Construção de edifícios,2021-08-16,SPE_PROVAVEL
546,16682639000215,EMPREENDIMENTOS IMOBILIARIOS NEZGODA LTDA,,SANTO ANTONIO DA PLATINA,PR,Incorporação de empreendimentos imobiliários,2012-12-21,SPE_PROVAVEL


In [ ]:
# 1. Função para buscar QSA

In [12]:
import requests
import time

def buscar_qsa(cnpj):
    url = f"https://minhareceita.org/{cnpj}"
    
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        socios = data.get("qsa", [])
        resultado = []

        for s in socios:
            resultado.append({
                "cnpj": cnpj,
                "socio_nome": s.get("nome_socio"),
                "qualificacao": s.get("qualificacao_socio")
            })

        return resultado

    except Exception as e:
        print(f"Erro ao buscar QSA {cnpj}: {e}")
        return []

In [ ]:
# 2. Montar base de sócios

In [13]:
def montar_base_socios(df_empresas):
    socios_total = []

    for i, row in df_empresas.iterrows():
        cnpj = row["CNPJ"]

        socios = buscar_qsa(cnpj)
        socios_total.extend(socios)

        if i % 10 == 0:
            print(f"Processados {i} CNPJs")

        time.sleep(0.5)

    return socios_total

In [ ]:
# 3. Criar grafo (agrupamento)

In [14]:
import pandas as pd

def agrupar_por_socios(lista_socios):
    df = pd.DataFrame(lista_socios)

    # remove sócios vazios
    df = df[df["socio_nome"].notna()]

    # agrupar
    grupos = df.groupby("socio_nome")["cnpj"].apply(list).reset_index()

    # filtrar sócios com mais de 1 empresa
    grupos["qtd_empresas"] = grupos["cnpj"].apply(len)
    grupos = grupos[grupos["qtd_empresas"] > 1]

    return grupos.sort_values("qtd_empresas", ascending=False)

In [ ]:
# 4. Uso completo

In [15]:
df_empresas = buscar_incorporadoras_ativas_dataframe(uf="PR", limite_total=200)

socios = montar_base_socios(df_empresas)

df_grupos = agrupar_por_socios(socios)

print(df_grupos.head())

Buscando dados (UF=PR, limite_total=200)...
Coletados: 69
Coletados: 138
Coletados: 207
Processados 0 CNPJs
Processados 10 CNPJs
Processados 20 CNPJs
Processados 30 CNPJs
Processados 40 CNPJs
Processados 50 CNPJs
Processados 60 CNPJs
Processados 70 CNPJs
Processados 80 CNPJs
Processados 90 CNPJs
Processados 100 CNPJs
Processados 110 CNPJs
Processados 120 CNPJs
Processados 130 CNPJs
Processados 140 CNPJs
Processados 150 CNPJs
Processados 160 CNPJs
Processados 170 CNPJs
Processados 180 CNPJs
Processados 190 CNPJs
Processados 200 CNPJs
                               socio_nome  \
0  A YOSHII ENGENHARIA E CONSTRUCOES LTDA   
1            A. YOSHII PARTICIPACOES LTDA   
2             ADRIANA GUIMARAES DEMETERCO   
3     AJG PARTICIPACOES IMOBILIARIAS LTDA   
4             ALEXANDRE CIRINO DOS SANTOS   

                                               cnpj  qtd_empresas  
0  [10910748004091, 10910748004091, 10910748004091]             3  
1  [10910748004091, 10910748004091, 10910748004091]   

In [16]:
socios

[{'cnpj': '29959934000137',
  'socio_nome': 'GERMANO BASKO',
  'qualificacao': 'Sócio'},
 {'cnpj': '29959934000137',
  'socio_nome': 'JULIO OSCAR CATALANO',
  'qualificacao': 'Sócio-Administrador'},
 {'cnpj': '05727088000161',
  'socio_nome': 'SAMUEL DE PAULA',
  'qualificacao': 'Sócio-Administrador'},
 {'cnpj': '10910748004091',
  'socio_nome': 'A YOSHII ENGENHARIA E CONSTRUCOES LTDA',
  'qualificacao': 'Sócio'},
 {'cnpj': '10910748004091',
  'socio_nome': 'A. YOSHII PARTICIPACOES LTDA',
  'qualificacao': 'Sócio'},
 {'cnpj': '10910748004091',
  'socio_nome': 'APARECIDO SIQUEIRA',
  'qualificacao': 'Administrador'},
 {'cnpj': '10910748004091',
  'socio_nome': 'ATSUSHI YOSHII',
  'qualificacao': 'Administrador'},
 {'cnpj': '10910748004091',
  'socio_nome': 'EVANDRO ANDERSON SANT ANA ZAGATTO',
  'qualificacao': 'Administrador'},
 {'cnpj': '10910748004091',
  'socio_nome': 'LEONARDO MAKOTO YOSHII',
  'qualificacao': 'Administrador'},
 {'cnpj': '10910748004091',
  'socio_nome': 'LUIZ ROGER

In [17]:
df_grupos

,socio_nome,cnpj,qtd_empresas
0,A YOSHII ENGENHARIA E CONSTRUCOES LTDA,"[10910748004091, 10910748004091, 10910748004091]",3
1,A. YOSHII PARTICIPACOES LTDA,"[10910748004091, 10910748004091, 10910748004091]",3
2,ADRIANA GUIMARAES DEMETERCO,"[00650202000189, 00650202000189, 00650202000189]",3
3,AJG PARTICIPACOES IMOBILIARIAS LTDA,"[42908452000116, 42908452000116, 42908452000116]",3
4,ALEXANDRE CIRINO DOS SANTOS,"[17079502000152, 17079502000152, 17079502000152]",3
...,...,...,...
163,VALMIR SOUZA,"[13762095000122, 13762095000122, 13762095000122]",3
164,VITOR BATISTELLA DE GODOES,"[41999763000175, 41999763000175, 41999763000175]",3
165,VOLMAR SANDER,"[30750259000110, 30750259000110, 30750259000110]",3
166,WANDER THIAGO RONCAGLIO,"[14978947000186, 14978947000186, 14978947000186]",3
